In [ ]:
import pandas as pd
import numpy as np
from matplotlib.ticker import LogFormatter
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import json
import sys
import plategig
import re

# Inputs are resolved relative to this notebook directory; run from si_figures/s02_mic/.
PROJECT_PATH = Path.cwd()
PROJECT_ID = '.'
(PROJECT_PATH / PROJECT_ID / 'figures').mkdir(parents=True, exist_ok=True)

#### Step 1: Read Plate Information

In [ ]:
# Read the Excel file into a DataFrame
df_plate_info = pd.read_excel(PROJECT_PATH / PROJECT_ID / 'data' / 'PlateInfo_250508_250510_250407.xlsx', engine='openpyxl')
df_plate_info.rename(columns={'Plate': 'Plate_ID'}, inplace=True)
df_plate_info

Collect unique plate IDs

#### Step 2: Read the raw OD data

In [ ]:
df_OD_raw = pd.read_excel(PROJECT_PATH / PROJECT_ID / 'data' /'ODFinal_250508_250510_250407.xlsx')
# drop last nan column, fix to input mistake
df_OD_raw = df_OD_raw.iloc[:,:-4]
df_OD_raw.tail()

In [ ]:
df = plategig.static.convert_OD_plate_to_long(df_OD_raw, df_plate_info)
df

Calculate median background of all plates to impute plates with missing media-only wells.

In [ ]:
df_plate_info[df_plate_info['Strain']=='Media Only']

In [ ]:
median_background_all_plates = plategig.static.calc_median_background_all_plates(
    df, df_plate_info, plot=True)
# Median background across all plates, used to impute plates with missing or inconsistent media-only wells.
print(median_background_all_plates)

In [ ]:
plategig.static.plot_single_plate_media_only_wells(df, df_plate_info, plate_id=1, column_name='OD')

Apply background correction for all plates

In [ ]:
df_bc = df.copy()
df_bc['OD_final'] = df_bc['OD'] - median_background_all_plates

#### Evaluate growth metrics

In [ ]:
df_bc

In [ ]:
# Prep the analysis dataframe
df_analysis = pd.merge(df_plate_info, df_bc, on=['Plate_ID', 'Well'])
df_analysis = df_analysis[['Experiment','Strain', 'Culture', 'Replicate', 'Antibiotic',
                           'Dose', 'Plate_ID', 'Well', 'Row', 'Column',
                           'OD', 'OD_final']]
# Remove control wells
df_analysis = df_analysis[~df_analysis['Strain'].isin(['Media Only','Cells Only'])]
# Keep group names
df_analysis['Group'] = df_analysis.Strain
# Define each culture of as a separate strain
# df_analysis['Strain'] = df_analysis.Strain + df_analysis.Culture.astype(int).astype(str)
df_analysis


In [ ]:
df_analysis['Strain'].value_counts()

In [ ]:
unique_groups = df_analysis['Group'].unique().tolist()
print(unique_groups)

In [ ]:
### Double check
unique_columns = df_analysis.columns.tolist()

# Print the list of unique columns
print(unique_columns)

#
unique_values = df_analysis['Strain'].unique()

# Print the unique values
print(unique_values)

In [ ]:
antibiotics = df_analysis['Antibiotic'].dropna().unique()
antibiotics

In [ ]:
df_analysis.query("Antibiotic == 'Cefotaxime' & Strain == 'NEB10B'")

In [ ]:

# Adjust the figure size as needed
fig, ax = plt.subplots(figsize=(3, 2))
plategig.static.plot_dose_response_curve_errorbar(df_analysis, 'NEB10B', 'Cefotaxime', strain_colors={'P1':'black'}, ax=ax)

In [ ]:
# Plot all dose-response curves in the df_analysis tables (might take too long)
for drug in antibiotics:
    if not pd.isna(drug):
        plategig.static.plot_od_final_for_selected_antibiotic(
            df_analysis,
            plategig.static.plot_dose_response_curve_errorbar,
            drug,
            strain_colors={})

In [ ]:
valid_combinations = plategig.static.prep_valid_combinations(
    df_analysis,
    multiplex=['Strain', 'Antibiotic'],
    ic50_threshold=0.5,
    mic_threshold=0.05)
valid_combinations

In [ ]:
growth_features = plategig.static.apply_phenotyper(df_analysis, valid_combinations)
growth_features = plategig.static.cap_growth_features_within_experiment_range(growth_features)
growth_features

In [ ]:
# growth_features['group'] = growth_features['Strain'].apply(lambda x: re.split(r'[1-9]', x)[0])
# growth_features

In [ ]:
strains = list(set(df_analysis['Strain'].str.lower()) - set(['media only', 'cells only', np.nan]))
print(strains)

In [ ]:
antibiotics = list(set(df_analysis['Antibiotic'].str.lower()) - set(['media only', 'cells only', np.nan]))
antibiotics

In [ ]:
strain_colors = {strain:color for strain, color in zip(strains, sns.color_palette('muted')[:len(strains)])}


In [ ]:
num_drugs = growth_features['Antibiotic'].nunique()
num_experiments = growth_features.shape[0]
num_cols = 6
num_rows = num_experiments//num_cols + (num_experiments % num_cols > 0)
# num_cols = num_experiments//num_drugs
# fig, axes = plt.subplots(num_experiments//num_replicates, num_replicates,
#                        figsize=(num_replicates*3, num_experiments*2), dpi=90)
fig, axes = plt.subplots(num_rows, num_cols,
                       figsize=(num_cols*3, num_rows*2), dpi=300)

for ix, row in growth_features.iterrows():
    plategig.static.plot_dose_response_curve_fit(df_analysis,
                                                growth_features=growth_features,
                                                strain_colors=strain_colors,
                                                strain=row['Strain'],
                                                antibiotic=row['Antibiotic'],
                                                ax=axes.flat[ix])
    
    if row['Status'] == 'FAIL':
        axes.flat[ix].text(0.5, 0.5, f'FAIL', color='red', fontsize=12, fontweight='bold',
                        ha='center', va='center', transform=axes.flat[ix].transAxes)
        # gray out the entire axes
        axes.flat[ix].set_facecolor('lightgray')
    if ix % num_cols != 0:
        axes.flat[ix].set_yticklabels([])
# Hide the rest of the axes
for ix in range(num_experiments, num_cols*num_rows):
    axes.flat[ix].axis('off')
    
fig.tight_layout()
# set empty space between subplots
fig.subplots_adjust(wspace=0.05)
fig.savefig(PROJECT_PATH / PROJECT_ID / 'figures' / f'IC50 fits.png', dpi=300, bbox_inches='tight')

In [ ]:
unique_columns = growth_features.columns.unique()
print(unique_columns)

In [ ]:
#### Graphes the IC50 values for each with a confindence interval
    ### looks like it pulls growth_features

final_chart = alt.vconcat()
for drug in sorted(growth_features['Antibiotic'].unique()):
    select_cols = ['Strain','Antibiotic','Status','IC50','IC50_ci_lower','IC50_ci_upper', 'insufficient_drug']
    filtered_growth_features = growth_features[select_cols]
    filtered_growth_features = filtered_growth_features.query(f'Antibiotic == "{drug}" & Status == "PASS"')
    filtered_growth_features['StrainGroup'] = filtered_growth_features['Strain']#.str.extract(r'(\D+)')

    base = alt.Chart(filtered_growth_features).encode(
        x=alt.X('Strain', axis=alt.Axis(labelAngle=-45, labelLimit=0)),
    ).properties(width=700, height=300)

    ic50 = base.encode(
        y=alt.Y('IC50:Q', scale=alt.Scale(type="log")),
        color=alt.Color("StrainGroup:N")
    ).mark_point(filled=True, size=200)

    error_bars = base.mark_errorbar(ticks=True, size=10, thickness=2).encode(
        alt.Y("IC50_ci_lower:Q", scale=alt.Scale(type="log")).title('IC50'),
        alt.Y2("IC50_ci_upper:Q"),
        color='StrainGroup'
    )

    # Create a dagger symbol on the bars with insufficient_drug flag is true
    dagger = base.mark_text(align='center', baseline='middle', fontSize=14, color='black', fontWeight='bold', dy=-20, dx=0).encode(
        y=alt.Y('IC50'),
        text=alt.condition(
            alt.datum.insufficient_drug, if_true=alt.value("‡"), if_false=alt.value("")
            )
    )

    layered_chart = alt.layer(ic50, error_bars, dagger).properties(title=drug)
    final_chart &= layered_chart

final_chart = final_chart.resolve_scale(y='independent', x='independent').configure_axis(
    labelFontSize=16,
    titleFontSize=16,
).configure_title(
    fontSize=16,
    anchor='middle',
).configure_legend(
    titleFontSize=16,
    labelFontSize=16,
    labelLimit=0,
    symbolLimit=50,
)

final_chart = final_chart.configure_title(anchor='middle')
final_chart.save(PROJECT_PATH / PROJECT_ID / 'figures' / f'IC50 combined.png', scale_factor=2)
final_chart.show()

In [ ]:
#### Graphs the MIC values for each with a confidence interval
    ### looks like it pulls growth_features

final_chart = alt.vconcat()
for drug in sorted(growth_features['Antibiotic'].unique()):
    select_cols = ['Strain','Antibiotic','Status','MIC','MIC_ci_lower','MIC_ci_upper', 'insufficient_drug']
    filtered_growth_features = growth_features[select_cols]
    filtered_growth_features = filtered_growth_features.query(f'Antibiotic == "{drug}" & Status == "PASS"')
    filtered_growth_features['StrainGroup'] = filtered_growth_features['Strain']#.str.extract(r'(\D+)')

    base = alt.Chart(filtered_growth_features).encode(
        x=alt.X('Strain', axis=alt.Axis(labelAngle=-45, labelLimit=0)),
    ).properties(width=700, height=300)

    mic = base.encode(
        y=alt.Y('MIC:Q', scale=alt.Scale(type="log")),
        color=alt.Color("StrainGroup:N")
    ).mark_point(filled=True, size=200)

    error_bars = base.mark_errorbar(ticks=True, size=10, thickness=2).encode(
        alt.Y("MIC_ci_lower:Q", scale=alt.Scale(type="log")).title('MIC'),
        alt.Y2("MIC_ci_upper:Q"),
        color='StrainGroup'
    )

    # Create a dagger symbol on the bars with insufficient_drug flag is true
    dagger = base.mark_text(align='center', baseline='middle', fontSize=14, color='black', fontWeight='bold', dy=-20, dx=0).encode(
        y=alt.Y('MIC'),
        text=alt.condition(
            alt.datum.insufficient_drug, if_true=alt.value("‡"), if_false=alt.value("")
            )
    )

    layered_chart = alt.layer(mic, error_bars, dagger).properties(title=drug)
    final_chart &= layered_chart

final_chart = final_chart.resolve_scale(y='independent', x='independent').configure_axis(
    labelFontSize=16,
    titleFontSize=16,
).configure_title(
    fontSize=16,
    anchor='middle',
).configure_legend(
    titleFontSize=16,
    labelFontSize=16,
    labelLimit=0,
    symbolLimit=50,
)

final_chart = final_chart.configure_title(anchor='middle')
final_chart.save(PROJECT_PATH / PROJECT_ID / 'figures' / f'MIC combined.png', scale_factor=2)
final_chart.show()

---
### Reproducibility archive

The cells below derive the summary table and render the single-panel figure from the deposited plate data:

1. Export `growth_features` (the fitted IC50/MIC table) to CSV as source data.
2. Render a single-panel MIC figure (strain on the x-axis, coloured by antibiotic) matching the manuscript's `figure_s02.png` layout, written to `figures/MIC_single_panel_reconstruction.png`. The faceted alternative in `reference/` is drawn from the same table.


In [ ]:
growth_features.to_csv(PROJECT_PATH / PROJECT_ID / 'data' / 'growth_features.csv', index=False)
growth_features.shape


In [ ]:
strain_order = [
    'NEB10B', 'TEM-1', 'TEM-1-CML',
    'E104K-R164N-M182T-E240K', 'L21P-E104K-R164N-E240K-T265M',
    'Q39K-E104K-R164S-E240K-T265M', 'E104K-R164N-E240K-T265M',
    'Q39K-E104K-R164N-E240K', 'Q39K-E104K-R164N-M182T-E240K',
    'L21P-E104K-R164N-M182T-E240K',
]
plot_df = growth_features[growth_features['Strain'].isin(strain_order)].copy()
plot_df['Strain'] = pd.Categorical(plot_df['Strain'], categories=strain_order, ordered=True)
plot_df = plot_df.sort_values(['Strain'])

antibiotics_order = sorted(plot_df['Antibiotic'].unique())
palette = dict(zip(antibiotics_order, sns.color_palette('tab10', n_colors=len(antibiotics_order))))

fig, ax = plt.subplots(figsize=(11, 7))
rng = np.random.default_rng(20250508)  # PlateInfo/ODFinal date stamp, for a reproducible jitter only
for ab in antibiotics_order:
    sub = plot_df[plot_df['Antibiotic'] == ab].dropna(subset=['MIC'])
    if sub.empty:
        continue
    x = sub['Strain'].cat.codes.to_numpy().astype(float)
    x = x + rng.uniform(-0.08, 0.08, size=len(x))
    yerr_lower = (sub['MIC'] - sub['MIC_ci_lower']).clip(lower=0)
    yerr_upper = (sub['MIC_ci_upper'] - sub['MIC']).clip(lower=0)
    ax.errorbar(x, sub['MIC'], yerr=[yerr_lower, yerr_upper], fmt='o', ms=7,
                color=palette[ab], ecolor=palette[ab], capsize=3, label=ab, alpha=0.9)

ax.set_yscale('log')
ax.set_xticks(range(len(strain_order)))
ax.set_xticklabels(strain_order, rotation=45, ha='right')
ax.set_xlabel('Strain')
ax.set_ylabel('MIC (\u00b5g/mL)')
ax.set_title('MIC Values by Strain and Antibiotic (Median \u00b1 Range) -- reconstruction, see README')
ax.legend(title='Antibiotic', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, which='both', axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(PROJECT_PATH / PROJECT_ID / 'figures' / 'MIC_single_panel_reconstruction.png', dpi=200, bbox_inches='tight')
plt.show()
